In [ ]:
from pathlib import Path
import re
import unicodedata
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
import geopandas as gpd

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_PATTERNS = {"Moov": "moov.csv", "Togocom": "Togocom.csv", "Telecom": "file-Agences*.csv", "CANAL+": "canalplus.csv", "Data center": "datacenter.csv", "Mobile Money": "mobile money.csv"}


def find_source(pattern):
    matches = list(RAW_DIR.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"Fichier introuvable : {pattern}")
    return matches[0]


def read_csv_safe(path):
    for encoding in ("utf-8-sig", "latin1"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Encodage impossible : {path.name}")


def clean_text(value):
    if not isinstance(value, str):
        return value
    try:
        return value.encode("latin1").decode("utf-8") if "Ã" in value else value
    except UnicodeError:
        return value


def normalize_key(value):
    value = clean_text(str(value)).upper()
    return re.sub(r"[^A-Z0-9]", "", unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode())


frames = []
for service, pattern in SOURCE_PATTERNS.items():
    path = find_source(pattern)
    frame = read_csv_safe(path)
    frame.columns = [str(column).strip() for column in frame.columns]
    for column in frame.select_dtypes(include="object"):
        frame[column] = frame[column].map(clean_text)
    frame["service"] = service
    frame["source_file"] = path.name
    frames.append(frame)

raw_data = pd.concat(frames, ignore_index=True, sort=False)
geometry = gpd.GeoSeries.from_wkt(raw_data["geometry"], on_invalid="ignore")
raw_data["lon"] = geometry.x
raw_data["lat"] = geometry.y
data_points = gpd.GeoDataFrame(raw_data, geometry=geometry, crs="EPSG:4326").dropna(subset=["lat", "lon"]).copy()
data_points["region_key"] = data_points["region_nom_bdd"].map(normalize_key)
data_points["commune_key"] = data_points["commune_nom_bdd"].map(normalize_key)
data_points["nom"] = data_points.get("etab_nom", data_points["service"]).fillna(data_points["service"])

# Lecture XML directe : le classeur RGPH contient une feuille de styles incompatible avec openpyxl.
population_file = RAW_DIR / "Population_residente_par_dcoupage_administratif_et_par_sexe.xlsx"
with zipfile.ZipFile(population_file) as archive:
    worksheet = ET.fromstring(archive.read("xl/worksheets/sheet1.xml"))
namespace = {"x": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
rows = [["".join(cell.itertext()).strip() for cell in row][:4] for row in worksheet.findall(".//x:sheetData/x:row", namespace)]
population_raw = pd.DataFrame(rows[1:], columns=["name", "sex", "unit", "pop"])
population_raw["pop"] = pd.to_numeric(population_raw["pop"], errors="coerce")
population = population_raw[population_raw["sex"].astype(str).str.casefold().eq("total")].copy()
population["commune_key"] = population["name"].map(normalize_key)
population = population.groupby("commune_key", as_index=False)["pop"].max()

data_points.drop(columns="geometry").to_csv(PROCESSED_DIR / "data_points.csv", index=False, encoding="utf-8")
data_points.to_file(PROCESSED_DIR / "data_points.geojson", driver="GeoJSON")
population.to_csv(PROCESSED_DIR / "population.csv", index=False, encoding="utf-8")

print(f"Points nettoyes : {len(data_points):,}")
print(f"Communes avec population : {population['commune_key'].nunique():,}")
print(data_points["service"].value_counts())